Chapter 2: Working with Text Data

2.2 Tokenizing text

In [17]:
import os
import urllib.request

if not os.path.exists('the-verdict.txt'):
    url = ("https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/refs/heads/main/ch02/"
    "01_main-chapter-code/the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

In [18]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [19]:
raw_text

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even 

In [20]:
len(raw_text)

20479

In [21]:
import re

text = "Hello, world. This is a test."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test.']


In [22]:
result = re.split(r'([,.]|\s)', text)

In [23]:
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [24]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', 'is', 'a', 'test', '.']


In [25]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
result = [item.strip() for item in result if item.strip()]
preprocessed = result

In [26]:
len(preprocessed)

4690

In [27]:
preprocessed[:10]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius']

2.3 Converting tokens into token IDs

In [28]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [29]:
vocab = {token:integer for integer,token in enumerate(all_words)}

vocab

{'!': 0,
 '"': 1,
 "'": 2,
 '(': 3,
 ')': 4,
 ',': 5,
 '--': 6,
 '.': 7,
 ':': 8,
 ';': 9,
 '?': 10,
 'A': 11,
 'Ah': 12,
 'Among': 13,
 'And': 14,
 'Are': 15,
 'Arrt': 16,
 'As': 17,
 'At': 18,
 'Be': 19,
 'Begin': 20,
 'Burlington': 21,
 'But': 22,
 'By': 23,
 'Carlo': 24,
 'Chicago': 25,
 'Claude': 26,
 'Come': 27,
 'Croft': 28,
 'Destroyed': 29,
 'Devonshire': 30,
 'Don': 31,
 'Dubarry': 32,
 'Emperors': 33,
 'Florence': 34,
 'For': 35,
 'Gallery': 36,
 'Gideon': 37,
 'Gisburn': 38,
 'Gisburns': 39,
 'Grafton': 40,
 'Greek': 41,
 'Grindle': 42,
 'Grindles': 43,
 'HAD': 44,
 'Had': 45,
 'Hang': 46,
 'Has': 47,
 'He': 48,
 'Her': 49,
 'Hermia': 50,
 'His': 51,
 'How': 52,
 'I': 53,
 'If': 54,
 'In': 55,
 'It': 56,
 'Jack': 57,
 'Jove': 58,
 'Just': 59,
 'Lord': 60,
 'Made': 61,
 'Miss': 62,
 'Money': 63,
 'Monte': 64,
 'Moon-dancers': 65,
 'Mr': 66,
 'Mrs': 67,
 'My': 68,
 'Never': 69,
 'No': 70,
 'Now': 71,
 'Nutley': 72,
 'Of': 73,
 'Oh': 74,
 'On': 75,
 'Once': 76,
 'Only': 77,
 '

In [30]:
vocab["Jack"]

57

In [40]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text



In [41]:
tokenizer = SimpleTokenizerV1(vocab)

In [42]:
text = """It's the last he painted, you know,"
          Mrs. Gisburn said with pardonable pride."""

In [44]:
ids = tokenizer.encode(text)
print(ids)

[56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [45]:
tokenizer.decode(ids)

'It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

2.4 Adding special context tokens

In [46]:
text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

KeyError: 'Hello'

In [61]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [62]:
print(len(vocab.items()))

1132


In [63]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [64]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [65]:
tokenizer = SimpleTokenizerV2(vocab)

In [68]:
text = "Hello, do you like tea. is this-- a test?"

tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 7, 584, 999, 6, 115, 1131, 10]

In [69]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea. is this -- a <|unk|>?'

2.5 BytePair encoding

BPE is used by GPT1, GPT2, GPT3, GPT4 

In [70]:
import tiktoken

In [ ]:
#!pip install tiktoken==0.14.0

In [72]:
tokenizer = tiktoken.get_encoding("gpt2")

In [73]:
tokenizer.encode("Hello world")

[15496, 995]

In [74]:
tokenizer.decode(tokenizer.encode("Hello world"))

'Hello world'

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> Infdjkhkwerhfkuerwhfkuerf the sunlit terraces"
     "of someunknownPlace."
)

# fdjkhkwerhfkuerwhfkuerf
tokenizer.encode(text, allowed_special={"<|endoftext|>"})

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 220,
 50256,
 4806,
 28241,
 14636,
 74,
 15448,
 71,
 69,
 74,
 15573,
 1929,
 69,
 74,
 15573,
 69,
 262,
 4252,
 18250,
 8812,
 2114,
 1659,
 617,
 34680,
 27271,
 13]

2.6 Data sampling with a sliding window

LLM ca not receive all tokens at once, you need to feed it to LLM as chunks

It is the topic of this sublesson

In [82]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [85]:
enc_sample = enc_text[50:]
len(enc_sample)

5095

In [89]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [90]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [91]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [92]:
import torch

In [93]:
torch.__version__

'2.15.0'

In [98]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
# ex 1
# ex 2

# ex 3
# ex 4

# ex 5
# ex 6

# drop_last=True
# ex 7

In [96]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [94]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=4, shuffle=False
    # stride=4 no overlap across batches
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [106]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[1807, 3619,  402,  271]]), tensor([[ 3619,   402,   271, 10899]])]


In [107]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


2.7 Creating token embeddings

IDs are converted to embeddings that are represented like vector of floating point numbers that be then optimized

In [108]:
inputs_ids = torch.tensor([2, 3, 5, 1])

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)  #vocab_size - tokenizer size, output_dim - emnbedding size

In [117]:
print(embedding_layer.weight)

Parameter containing:
tensor([[-0.1115,  0.1204, -0.3696],
        [-0.2404, -1.1969,  0.2093],
        [-0.9724, -0.7550,  0.3239]], requires_grad=True)


In [111]:
embedding_layer(torch.tensor([3]))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)

In [112]:
embedding_layer(torch.tensor([2]))

tensor([[ 1.2753, -0.2010, -0.1606]], grad_fn=<EmbeddingBackward0>)

In [113]:
embedding_layer(inputs_ids)

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)

2.8 Encoding word positions

In [ ]:
vocab_size = 50257
output_dim = 256
# gpt2 was trained using output_dim=1024?

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [119]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [120]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [121]:
token_embeddings = token_embedding_layer(inputs)
token_embeddings.shape

torch.Size([8, 4, 256])

In [122]:
token_embeddings[0, 0]

tensor([-1.2679e-01,  4.5306e-01, -7.6017e-01, -1.0059e+00,  3.5732e-01,
        -4.4823e-01,  4.0548e-01, -1.4911e+00,  1.0657e+00,  3.9532e-01,
         5.7309e-01, -7.7677e-01, -1.5741e+00, -2.3430e-01,  5.3594e-01,
        -6.5515e-01,  1.4588e+00, -3.6530e-01, -4.0372e-02,  1.9042e-01,
         6.8024e-01, -8.6577e-01,  3.5541e-01,  4.1035e-01, -1.5947e+00,
        -2.6210e-02, -1.4804e+00, -1.0301e+00, -1.2018e+00,  1.2016e+00,
        -2.0944e-01, -5.3766e-02,  1.6373e-01, -5.1866e-01,  5.2069e-01,
        -1.0986e+00,  4.8871e-01,  4.3105e-01,  8.6485e-02, -1.1179e+00,
         9.2041e-02,  4.5699e-01, -1.4753e+00, -4.4934e-01, -8.3517e-01,
        -1.1789e+00,  4.4643e-01,  1.7063e+00, -2.9543e-01,  5.1638e-01,
         4.2293e-01,  2.7946e-01, -2.4484e+00,  3.0050e-01, -8.8245e-01,
         2.9455e-01,  2.1654e-01, -9.2025e-01,  2.2604e-01, -8.0408e-01,
         6.7722e-01,  8.9504e-01, -5.4177e-01, -8.7920e-01, -1.6062e+00,
        -8.2329e-01,  1.1738e-01, -7.6778e-01,  9.9

In [ ]:
# There are many ways if implementing positonal embeddings like rotational positional embeddings, Llama 3.2 used
# rope embeddings
# GPT2 just the 2nd embedding layer
context_length = max_length # how many inputs are supported
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [125]:
torch.arange(max_length) # placeholder for positions

tensor([0, 1, 2, 3])

In [126]:
pos_embedding_layer.weight

Parameter containing:
tensor([[ 1.9121,  1.6565,  0.3007,  ...,  0.5781,  0.5038, -0.4022],
        [-1.4743, -0.3636, -1.6835,  ..., -2.1543,  1.6986, -1.4720],
        [ 0.1597,  1.1583,  1.1767,  ..., -0.7823, -0.7058, -1.1376],
        [ 0.5423,  0.9308,  0.3779,  ..., -0.1079, -0.0839,  0.8274]],
       requires_grad=True)

In [127]:
pos_embedding_layer(torch.arange(max_length))

tensor([[ 1.9121,  1.6565,  0.3007,  ...,  0.5781,  0.5038, -0.4022],
        [-1.4743, -0.3636, -1.6835,  ..., -2.1543,  1.6986, -1.4720],
        [ 0.1597,  1.1583,  1.1767,  ..., -0.7823, -0.7058, -1.1376],
        [ 0.5423,  0.9308,  0.3779,  ..., -0.1079, -0.0839,  0.8274]],
       grad_fn=<EmbeddingBackward0>)

In [124]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [129]:
print(token_embeddings[0] + pos_embeddings)

print(token_embeddings + pos_embeddings)

tensor([[ 1.7853,  2.1095, -0.4595,  ...,  0.0107,  1.6831, -0.5362],
        [-1.4499, -0.0399, -1.2033,  ..., -2.4490,  2.1009, -0.4964],
        [-0.1016,  1.0871,  1.7139,  ..., -1.7677, -0.7123, -1.4182],
        [ 2.3651,  1.7130,  1.7235,  ...,  1.5690,  0.1998,  0.6486]],
       grad_fn=<AddBackward0>)
tensor([[[ 1.7853,  2.1095, -0.4595,  ...,  0.0107,  1.6831, -0.5362],
         [-1.4499, -0.0399, -1.2033,  ..., -2.4490,  2.1009, -0.4964],
         [-0.1016,  1.0871,  1.7139,  ..., -1.7677, -0.7123, -1.4182],
         [ 2.3651,  1.7130,  1.7235,  ...,  1.5690,  0.1998,  0.6486]],

        [[ 3.6793,  1.4374,  0.6705,  ..., -0.6073, -1.2815,  0.2389],
         [-1.6377, -0.3561, -1.8807,  ..., -2.4637,  1.9489, -1.2245],
         [-0.2759,  0.3818,  1.8327,  ...,  1.1926, -0.4849, -2.6000],
         [ 2.1370,  1.7698,  2.8638,  ..., -0.3780, -0.1250,  1.2979]],

        [[ 2.1166,  4.0321,  1.4811,  ...,  0.3052,  2.2195, -0.1540],
         [-2.9205,  0.0101, -2.2200,  ..., -2

In [ ]:
print(f'token_embeddings.shape: {token_embeddings.shape}')
print(f'pos_embeddings.shape: {pos_embeddings.shape}')
input_embeddings = token_embeddings + pos_embeddings #adding pos embedding to each batch separately
print(input_embeddings.shape)

token_embeddings.shape: torch.Size([8, 4, 256])
pos_embeddings.shape: torch.Size([4, 256])
torch.Size([8, 4, 256])
